# Session 12: PCA

In [2]:
import numpy as np

th = np.radians(30)
R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
z1 = np.array([-2.9, -1.7, -0.6, 0.5, 1.7, 3.0])
z2 = np.array([0.8, -0.9, 1.0, -0.8, 0.9, -0.6])
P = np.c_[z1, z2] @ R.T
P = P - P.mean(0)                                  # centred data
n = len(P)
lam, V = np.linalg.eigh(np.cov(P.T, bias=True))
u, v = V[:, 1], V[:, 0]                            # u = first principal axis, v perpendicular
if u[0] < 0: u = -u
if u[0] * v[1] - u[1] * v[0] < 0: v = -v           # v is u rotated by +90 degrees
print("angle of u:", np.degrees(np.arctan2(u[1], u[0])), " variances", lam[::-1])

f = lambda a: f"({a[0]:.3f},{a[1]:.3f})"
lines = []
lines.append(r"""\documentclass[border=6pt]{standalone}
\usepackage{tikz}
\usepackage{amsmath}
\usetikzlibrary{arrows.meta}
\begin{document}
\begin{tikzpicture}[scale=1.25,>=Stealth,font=\small]
\def\pu{red!70!black}
\def\pv{blue!70!black}
% axes
\draw[->,gray] (-4.1,0) -- (4.2,0) node[below left] {$x_1$};
\draw[->,gray] (0,-2.9) -- (0,3.0) node[below left] {$x_2$};""")
lines.append(f"% the two principal axes through the mean")
lines.append(rf"\draw[thick,\pu] {f(-4.0*u)} -- {f(4.0*u)};")
lines.append(rf"\draw[thick,dashed,\pv] {f(-2.6*v)} -- {f(2.6*v)};")
lines.append(rf"\node[\pu,above] at {f(3.6*u+np.array([-0.35,0.45]))} {{span$(u)$}};")
lines.append(rf"\node[\pv,left] at {f(2.45*v)} {{span$(u_\perp)$}};")
lines.append(rf"\draw[->,very thick,\pu] (0,0) -- {f(u)} node[below right] {{$u$}};")
lines.append(rf"\draw[->,very thick,\pv] (0,0) -- {f(v)} node[above left] {{$u_\perp$}};")

side = 0.2
def square(foot, d1, d2):
    p1, p2, p3 = foot + side*d1, foot + side*d1 + side*d2, foot + side*d2
    return rf"\draw[thin] {f(p1)} -- {f(p2)} -- {f(p3)};"

for i, p in enumerate(P):
    a = (u @ p) * u          # projection on span(u)
    b = (v @ p) * v          # projection on span(u_perp)
    lines.append(rf"\draw[dashed,\pu!60] {f(p)} -- {f(a)};")
    lines.append(rf"\draw[dashed,\pv!60] {f(p)} -- {f(b)};")
    if i in (4, 1):
        d_p = (p - a) / np.linalg.norm(p - a)
        lines.append(square(a, u * np.sign(u @ p) * -1 if False else (-u if u @ p > 0 else u), d_p))
        d_pb = (p - b) / np.linalg.norm(p - b)
        lines.append(square(b, (-v if v @ p > 0 else v), d_pb))
for i, p in enumerate(P):
    a = (u @ p) * u
    b = (v @ p) * v
    lines.append(rf"\fill[\pu] {f(a)} circle (1.6pt);")
    lines.append(rf"\fill[\pv] {f(b)} circle (1.6pt);")
for i, p in enumerate(P):
    lines.append(rf"\fill {f(p)} circle (2.2pt);")
# labels for the first point
p = P[4]
a = (u @ p) * u
lines.append(rf"\node[above right] at {f(p)} {{$x^{{(i)}}$}};")
lines.append(rf"\node[\pu,below right=-1pt] at {f(a)} {{$(u^Tx^{{(i)}})\,u$}};")
q = P[1]
b = (v @ q) * v
lines.append(rf"\node[\pv,right=3pt] at {f(b)} {{$(u_\perp^Tx^{{(j)}})\,u_\perp$}};")
lines.append(rf"\node[below left] at {f(q)} {{$x^{{(j)}}$}};")
lines.append(rf"\node[anchor=north west,align=left,font=\footnotesize] at (1.6,-1.7) {{{{\color{{red!70!black}}\rule[0.2ex]{{1.2em}}{{1.2pt}}}}\ projection onto span$(u)$\\[2pt] {{\color{{blue!70!black}}\rule[0.2ex]{{1.2em}}{{1.2pt}}}}\ projection onto span$(u_\perp)$}};")
lines.append(r"\end{tikzpicture}")
lines.append(r"\end{document}")
open("pca_projection_2d.tex", "w").write("\n".join(lines))


angle of u: 24.05468714208496  variances [4.00242117 0.66980106]


2904

In [1]:
"""
PCA as a projection onto the span of the leading eigenvectors.

Let X_c = X - mu be the centred data and S = X_c^T X_c / (n-1) the covariance
matrix with eigendecomposition S = V diag(lambda) V^T. For the first k
eigenvectors V_k = [v_1, ..., v_k] the map in three steps is

    scores         z   = V_k^T (x - mu)           coordinates in the eigenbasis
    projection     P   = V_k V_k^T                orthogonal projector of rank k
    reconstruction x^  = mu + P (x - mu) = mu + V_k z

The residual (I - P)(x - mu) is orthogonal to span(v_1..v_k). The script checks

    Pythagoras             |x - mu|^2 = |P (x - mu)|^2 + |(I - P)(x - mu)|^2
    variance of a score    Var(z_j)   = lambda_j, and the scores are uncorrelated
    reconstruction error   sum_i |x_i - x^_i|^2 / (n-1) = lambda_{k+1} + ... + lambda_m

and compares the eigenvectors with scikit-learn and with the SVD of X_c.

Part 1 uses 3 dimensions so that the geometry can be drawn. Part 2 uses the
64 dimensional digits data of scikit-learn.
"""
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA as SkPCA

SEED = 0


class PCA:
    def fit(self, X):
        self.mu = X.mean(axis=0)
        Xc = X - self.mu
        self.n = len(X)
        S = Xc.T @ Xc / (self.n - 1)
        lam, V = np.linalg.eigh(S)                      # eigh returns ascending eigenvalues
        order = np.argsort(lam)[::-1]
        self.lam, self.V = lam[order], V[:, order]
        # sign convention, the largest component of every eigenvector is positive
        signs = np.sign(self.V[np.abs(self.V).argmax(axis=0), np.arange(self.V.shape[1])])
        self.V = self.V * signs
        return self

    def scores(self, X, k):
        return (X - self.mu) @ self.V[:, :k]

    def reconstruct(self, X, k):
        return self.mu + self.scores(X, k) @ self.V[:, :k].T

    def projector(self, k):
        return self.V[:, :k] @ self.V[:, :k].T


# ---------------------------------------------------------------------------
# Part 1, three dimensions
# ---------------------------------------------------------------------------
def make_cloud(n=45):
    rng = np.random.default_rng(SEED)
    Q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
    z = rng.normal(size=(n, 3)) * np.array([2.0, 1.1, 0.6])
    return np.array([1.0, 0.5, 1.0]) + z @ Q.T


def equal_axes(ax, centre, half):
    ax.set_xlim(centre[0] - half, centre[0] + half)
    ax.set_ylim(centre[1] - half, centre[1] + half)
    ax.set_zlim(centre[2] - half, centre[2] + half)
    ax.set_box_aspect((1, 1, 1))


def fig_geometry(X, p, fname, k=2):
    Xh = p.reconstruct(X, k)
    Z = p.scores(X, 3)
    half = 1.1 * np.abs(X - p.mu).max()
    ext = 1.15 * np.abs(p.scores(X, 2)).max()
    fig = plt.figure(figsize=(17, 5.8))

    # (a) original coordinates
    ax = fig.add_subplot(1, 3, 1, projection="3d")
    a = np.linspace(-ext, ext, 2)
    A, B = np.meshgrid(a, a)
    plane = p.mu[:, None, None] + p.V[:, 0, None, None] * A + p.V[:, 1, None, None] * B
    ax.plot_surface(*plane, alpha=0.18, color="tab:blue", linewidth=0)
    for i in range(len(X)):
        ax.plot(*zip(X[i], Xh[i]), color="dimgrey", lw=1.0)
    ax.scatter(*X.T, s=16, color="k", label="data $x$")
    ax.scatter(*Xh.T, s=16, color="tab:red", label="projection $\\hat x$ on the plane")
    ax.scatter(*p.mu, s=60, color="gold", edgecolor="k", label="mean $\\mu$")
    for j, c in enumerate(["tab:red", "tab:green", "tab:purple"]):
        v = 2.0 * np.sqrt(p.lam[j]) * p.V[:, j]
        ax.quiver(*p.mu, *v, color=c, lw=2.2, arrow_length_ratio=0.12)
        ax.text(*(p.mu + 1.12 * v), f"$v_{j + 1}$", color=c, fontsize=13)
    equal_axes(ax, p.mu, half)
    d = 0.75 * p.V[:, 2] + 0.45 * p.V[:, 0] - 0.45 * p.V[:, 1]
    d = d / np.linalg.norm(d)
    ax.view_init(elev=np.degrees(np.arcsin(d[2])), azim=np.degrees(np.arctan2(d[1], d[0])))
    ax.set_title("original coordinates\nplane $=\\mu+\\mathrm{span}(v_1,v_2)$, residuals are orthogonal to it", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")

    # (b) eigenbasis coordinates, the plane is z3 = 0 and the projection drops z3
    ax = fig.add_subplot(1, 3, 2, projection="3d")
    h = 1.1 * np.abs(Z).max()
    g = np.linspace(-ext, ext, 2)
    G1, G2 = np.meshgrid(g, g)
    ax.plot_surface(G1, G2, 0 * G1, alpha=0.18, color="tab:blue", linewidth=0)
    for i in range(len(Z)):
        ax.plot([Z[i, 0]] * 2, [Z[i, 1]] * 2, [Z[i, 2], 0], color="dimgrey", lw=1.0)
    ax.scatter(*Z.T, s=16, color="k")
    ax.scatter(Z[:, 0], Z[:, 1], 0 * Z[:, 0], s=16, color="tab:red")
    ax.view_init(elev=22, azim=-58)
    ax.set_xlim(-h, h); ax.set_ylim(-h, h); ax.set_zlim(-h, h); ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$"); ax.set_zlabel("$z_3$")
    ax.set_title("coordinates $z=V^T(x-\\mu)$ in the eigenbasis\nthe projection sets $z_3$ to zero", fontsize=10)

    # (c) the retained coordinates
    ax = fig.add_subplot(1, 3, 3)
    ax.scatter(Z[:, 0], Z[:, 1], s=22, color="tab:red")
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
    ax.set_aspect("equal"); ax.set_xlabel("$z_1$ (PC 1)"); ax.set_ylabel("$z_2$ (PC 2)")
    ax.set_title(f"the 2 retained coordinates\nvariances {p.lam[0]:.2f} and {p.lam[1]:.2f}", fontsize=10)
    fig.suptitle("PCA maps $\\mathbb{R}^3$ onto the plane spanned by the two leading eigenvectors", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94]); fig.savefig(fname, dpi=130); plt.close(fig)


def checks(X, p, ks):
    Xc, n, m = X - p.mu, len(X), X.shape[1]
    print(f"orthonormal eigenvectors      max|V^T V - I| = {np.abs(p.V.T @ p.V - np.eye(m)).max():.1e}")
    sk = SkPCA(n_components=m).fit(X)
    # eigenvectors with a zero eigenvalue (constant pixels) are not unique, so only the others are compared
    nz = [j for j in range(m) if p.lam[j] > 1e-8]
    same = max(min(np.abs(sk.components_[j] - p.V[:, j]).max(), np.abs(sk.components_[j] + p.V[:, j]).max())
               for j in nz)
    sv = np.linalg.svd(Xc, compute_uv=False)
    print(f"against scikit-learn          max |v_j - v_j(sklearn)| = {same:.1e} (up to sign, {len(nz)} of {m} eigenvectors with nonzero eigenvalue)")
    print(f"against the SVD of X_c        max |s_j^2/(n-1) - lambda_j| = {np.abs(sv ** 2 / (n - 1) - p.lam).max():.1e}")
    for k in ks:
        P = p.projector(k)
        Pc, Rc = Xc @ P, Xc - Xc @ P
        Z = p.scores(X, k)
        resid_err = (Rc ** 2).sum() / (n - 1)
        print(f"k={k:2d}  P^2-P {np.abs(P @ P - P).max():.0e}  |V_k^T r| {np.abs(Rc @ p.V[:, :k]).max():.0e}  "
              f"Pythagoras {np.abs((Xc ** 2).sum(1) - (Pc ** 2).sum(1) - (Rc ** 2).sum(1)).max():.0e}  "
              f"cov(Z)-Lambda {np.abs(np.cov(Z.T).reshape(k, k) - np.diag(p.lam[:k])).max():.0e}  "
              f"error {resid_err:.4f} = sum of discarded lambda {p.lam[k:].sum():.4f}")


# ---------------------------------------------------------------------------
# Part 2, digits in 64 dimensions
# ---------------------------------------------------------------------------
def fig_digits(Xd, yd, p, fname_images, fname_spectrum):
    ks = [2, 5, 10, 20, 40]
    cols = 8
    rows = 2 + len(ks)
    fig, axes = plt.subplots(rows, cols, figsize=(11, 1.45 * rows))
    labels = ["eigenvectors $v_1\\dots v_8$", "original $x$"] + [f"$k={k}$" for k in ks]
    for j in range(cols):
        axes[0, j].imshow(p.V[:, j].reshape(8, 8), cmap="RdBu", vmin=-0.5, vmax=0.5)
        axes[0, j].set_title(f"$v_{j + 1}$", fontsize=9)
        axes[1, j].imshow(Xd[j].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
        for r, k in enumerate(ks):
            axes[2 + r, j].imshow(np.clip(p.reconstruct(Xd[j:j + 1], k).reshape(8, 8), 0, 16),
                                  cmap="gray_r", vmin=0, vmax=16)
    for r in range(rows):
        axes[r, 0].set_ylabel(labels[r], rotation=0, ha="right", va="center", fontsize=9)
        for j in range(cols):
            axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
    fig.suptitle("A digit is a point in $\\mathbb{R}^{64}$. Its projection onto span$(v_1..v_k)$ is a blurred digit", y=0.995)
    fig.tight_layout(); fig.savefig(fname_images, dpi=130); plt.close(fig)

    n, m = Xd.shape
    ks_all = np.arange(0, m + 1)
    err = np.array([((Xd - p.mu - (Xd - p.mu) @ p.projector(k)) ** 2).sum() / (n - 1) if k > 0
                    else ((Xd - p.mu) ** 2).sum() / (n - 1) for k in ks_all])
    theory = np.array([p.lam[k:].sum() for k in ks_all])
    Z2 = p.scores(Xd, 2)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
    axes[0, 0].semilogy(np.arange(1, m + 1), p.lam, "o-", ms=4)
    axes[0, 0].set_title("eigenvalues $\\lambda_j$ = variance along $v_j$"); axes[0, 0].set_xlabel("$j$")
    cum = np.cumsum(p.lam) / p.lam.sum()
    axes[0, 1].plot(np.arange(1, m + 1), cum, "o-", ms=4)
    for q in [0.9, 0.95]:
        kq = int(np.searchsorted(cum, q) + 1)
        axes[0, 1].axhline(q, color="grey", ls=":"); axes[0, 1].axvline(kq, color="grey", ls=":")
        axes[0, 1].text(kq + 1, q - 0.06, f"k={kq} keeps {q:.0%}", fontsize=9)
    axes[0, 1].set_title("cumulative share of the variance"); axes[0, 1].set_xlabel("$k$")
    axes[1, 0].semilogy(ks_all, np.maximum(theory, 1e-12), "-", label="$\\lambda_{k+1}+\\dots+\\lambda_{64}$")
    axes[1, 0].semilogy(ks_all, np.maximum(err, 1e-12), "k.", ms=6, label="measured reconstruction error")
    axes[1, 0].set_title("squared reconstruction error of the projection"); axes[1, 0].set_xlabel("$k$")
    axes[1, 0].legend()
    sc = axes[1, 1].scatter(Z2[:, 0], Z2[:, 1], c=yd, cmap="tab10", s=8)
    axes[1, 1].set_xlabel("$z_1$"); axes[1, 1].set_ylabel("$z_2$")
    axes[1, 1].set_title("the 64 dimensional digits in the plane of $v_1,v_2$")
    fig.colorbar(sc, ax=axes[1, 1], label="digit")
    fig.tight_layout(); fig.savefig(fname_spectrum, dpi=130); plt.close(fig)


if __name__ == "__main__":
    X = make_cloud()
    p = PCA().fit(X)
    print("Part 1, 3 dimensions, eigenvalues", np.round(p.lam, 3))
    checks(X, p, [1, 2])
    fig_geometry(X, p, "pca_geometry_3d.png")

    d = load_digits()
    Xd, yd = d.data, d.target
    pd_ = PCA().fit(Xd)
    print("\nPart 2, digits, 64 dimensions, first eigenvalues", np.round(pd_.lam[:5], 2))
    checks(Xd, pd_, [2, 10, 40])
    fig_digits(Xd, yd, pd_, "pca_digits_reconstruction.png", "pca_digits_spectrum.png")

Part 1, 3 dimensions, eigenvalues [5.328 0.879 0.22 ]
orthonormal eigenvectors      max|V^T V - I| = 6.7e-16
against scikit-learn          max |v_j - v_j(sklearn)| = 5.6e-16 (up to sign, 3 of 3 eigenvectors with nonzero eigenvalue)
against the SVD of X_c        max |s_j^2/(n-1) - lambda_j| = 8.9e-16
k= 1  P^2-P 7e-16  |V_k^T r| 3e-15  Pythagoras 3e-14  cov(Z)-Lambda 2e-15  error 1.0988 = sum of discarded lambda 1.0988
k= 2  P^2-P 7e-16  |V_k^T r| 3e-15  Pythagoras 3e-14  cov(Z)-Lambda 2e-15  error 0.2203 = sum of discarded lambda 0.2203

Part 2, digits, 64 dimensions, first eigenvalues [179.01 163.72 141.79 101.1   69.51]
orthonormal eigenvectors      max|V^T V - I| = 2.0e-15
against scikit-learn          max |v_j - v_j(sklearn)| = 6.1e-12 (up to sign, 61 of 64 eigenvectors with nonzero eigenvalue)
against the SVD of X_c        max |s_j^2/(n-1) - lambda_j| = 1.4e-13
k= 2  P^2-P 1e-16  |V_k^T r| 2e-14  Pythagoras 2e-12  cov(Z)-Lambda 1e-13  error 859.4230 = sum of discarded lambda 859.4